In [1]:
%matplotlib widget

from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from orcs.process import SpectralCube
import orb.utils.io

# -----------------------------
# User settings
# -----------------------------
CUBE_PATH = "/Volumes/RayPass/Research/SIGNALS/N604_SN4/3266414z.hdf5"

BASE_LINES = ["Halpha", "[NII]6583", "[NII]6548"]
MAX_COMPONENTS = 5
BIC_THRESHOLD = 10.0

# Masking threshold: replace this with whatever map you actually want to threshold on.
# A quick option is the deep frame. A better production option is a precomputed Halpha flux map.
FLUX_THRESHOLD = 0.0

# Your sky annulus
SKY_X = 331
SKY_Y = 1374
SKY_R_IN = 1
SKY_R_OUT = 40

# Fit setup
FMODEL = "sincgaussphased"
ALPHA_DEF = "1"
ALPHA_COV = np.pi / 20.0
SIGMA_GUESS = 10.0
VEL_STEP = 35.0   # spacing between initial velocity guesses for extra components

# Optional: hard-coded seed velocity if you do not have a velocity map yet
DEFAULT_VEL_GUESS = -180.0

OUTDIR = Path("multicomp_results")
OUTDIR.mkdir(exist_ok=True)

In [50]:
def fit_components(spec_corr):
    """
    Run an ORCS fit with one to four components
    """

    # 1 component fit
    fit1 = spec_corr.fit(
        ['Halpha','[NII]6583','[NII]6548'],
        fmodel='sincgaussphased',
        alpha_def='1',
        alpha_cov=np.pi/20,
        pos_def = ['1','1','1'],
        pos_cov = [-220],
        sigma_def = ['1','1','1'],
        sigma_cov = [10],
        nofilter=False
    )

    fit2 = spec_corr.fit(
        ['Halpha','[NII]6583','[NII]6548','Halpha','[NII]6583','[NII]6548'],
        fmodel='sincgaussphased',
        alpha_def='1',
        alpha_cov=np.pi/20,
        pos_def = ['1','1','1','2','2','2'],
        pos_cov = [-270,-180],
        sigma_def = ['1','1','1','2','2','2'],
        sigma_cov = [10,10],
        nofilter=False
    )

    fit3 = spec_corr.fit(
        ['Halpha','[NII]6583','[NII]6548','Halpha','[NII]6583','[NII]6548','Halpha','[NII]6583','[NII]6548'],
        fmodel='sincgaussphased',
        alpha_def='1',
        alpha_cov=np.pi/20,
        pos_def = ['1','1','1','2','2','2','3','3','3'],
        pos_cov = [-220,-200,-180],
        sigma_def = ['1','1','1','2','2','2','3','3','3'],
        sigma_cov = [10,10,10],
        nofilter=False
    )

    # fit4 = spec_corr.fit(
    #     ['Halpha','[NII]6583','[NII]6548','Halpha','[NII]6583','[NII]6548','Halpha','[NII]6583','[NII]6548','Halpha','[NII]6583','[NII]6548'],
    #     fmodel='sincgaussphased',
    #     alpha_def='1',
    #     alpha_cov=np.pi/20,
    #     pos_def = ['1','1','1','2','2','2','3','3','3','4','4','4'],
    #     pos_cov = [-220,-200,-180,-240],
    #     sigma_def = ['1','1','1','2','2','2','3','3','3','4','4','4'],
    #     sigma_cov = [10,10,10,10],
    #     nofilter=False
    # )

    return fit1, fit2, fit3


In [47]:
def choose_best_component_count(fits, bic_threshold=10.0):
    bic_dict = {n: fits[n]['BIC'] for n in range(0,len(fit_list))}
    delta_dict = {}

    best_n = 0
    for n in range(0, max(bic_dict.keys())):
        print(n)
        # if (n not in fits_by_n) or ((n + 1) not in fits_by_n):
        #     continue
        delta = bic_dict[n] - bic_dict[n+1]
        print(delta)
        delta_dict[(n,n+1)] = delta

        if delta < bic_threshold:
            best_n = n+1
        else:
            break
    
    return best_n, bic_dict, delta_dict



In [51]:
cube = SpectralCube(CUBE_PATH)

background = cube.get_spectrum_in_annulus(
    SKY_X, SKY_Y, SKY_R_IN, SKY_R_OUT,
    median=True,
    mean_flux=True,
)

x = 1070
y = 995

spec = cube.get_spectrum(int(x), int(y), 1, mean_flux=False)
spec_corr = spec.subtract_sky(background)

fits_by_n = {}

fit1, fit2, fit3 = fit_components(spec_corr)

fit_list = [fit1,fit2,fit3]

best_n, bic_dict, delta_dict = choose_best_component_count(fit_list)
best_fit = fit_list[best_n]





unknown|INFO| Cube is level 3
unknown|WARNING| error reading param from attributes polyref: Insufficient precision in available types to represent (79, 64, 15, 0, 64)
unknown|WARNING| error reading param from attributes polyref: Insufficient precision in available types to represent (79, 64, 15, 0, 64)
unknown|INFO| shape: (2048, 2064, 934)
unknown|INFO| wavenumber calibration: True
unknown|INFO| flux calibration: True
unknown|INFO| wcs calibration: True
unknown|WARNING| /opt/anaconda3/envs/orb3/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1095: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanmedian1d, axis, a, overwrite_input)

unknown|WARNING| error reading param from attributes polyref: Insufficient precision in available types to represent (79, 64, 15, 0, 64)
unknown|WARNING| No initial alpha given
unknown|WARNING| No initial alpha given
unknown|WARNING| No initial alpha given
unknown|WARNING| error reading param from attributes polyref: Insuff

TypeError: list indices must be integers or slices, not str

In [54]:
print(fit3)

=== Fit results ===
lines: ['H3', '[NII]6584', '[NII]6548', 'H3', '[NII]6584', '[NII]6548', 'H3', '[NII]6584', '[NII]6548'], fmodel: sincgaussphased
iterations: 4, fit time: 1.74e-01 s
number of free parameters: 18, BIC: -6.49331e+04, chi2: 5.07e+37
Velocity (km/s): [396.051(0) 396.051(0) 396.051(0) -741.174(0) -741.174(0) -741.174(0)
 456.975(0) 456.975(0) 456.975(0)] 
Flux: [5.96734e-12 +- nan 1.22092e-13 +- nan 7.07718e-15 +- nan
 3.04671e-12 +- nan 4.51362e-15 +- nan 7.23322e-15 +- nan
 5.16167e-15 +- nan 2.42973e-15 +- nan 5.89369e-15 +- nan]
Broadening (km/s): [12.5511 +- nan 12.5511 +- nan 12.5511 +- nan 8.60937 +- nan
 8.60937 +- nan 8.60937 +- nan 9.42231 +- nan 9.42231 +- nan
 9.42231 +- nan]
SNR (km/s): [inf inf inf inf inf inf inf inf inf]

